<a href="https://colab.research.google.com/github/Vaib-raksh/Intern-ML/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vaib-raksh/Intern-ML/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distributions- answer

I examined the distributions of the main Google Search Console metrics used in my lane: `gsc_impressions`, `gsc_clicks`, and `gsc_sum_position`.

The summary statistics show that these variables are heavily right-skewed (heavy-tailed). For example, the median number of impressions is only **16**, while the maximum is **40,084**. Similarly, at least half of the webpages receive **0 clicks**, but a few pages receive as many as **274 clicks**. This indicates that most webpages receive relatively little traffic, while a small number account for a large share of impressions and clicks.

Understanding these distributions is important before testing relationships because averages alone may not represent the typical webpage.

In [1]:
!pip install -q duckdb datasets huggingface_hub   #connecting the dataset- install the required libraries

In [2]:
# Read the HF_TOKEN from colab
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!" if HF_TOKEN else "Token not found")

#connect DuckDB to Hugging face
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

print("Connected successfully!")


Token loaded successfully!
Connected successfully!


In [3]:
DATASET = "hf://datasets/FlyRank/internship-warehouse" # the dataset path

In [4]:
distribution_df = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position
FROM read_parquet(
    '{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
""").df()

distribution_df.describe()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,gsc_impressions,gsc_clicks,gsc_sum_position
count,3.611061e+06,3.611061e+06,3.611061e+06
mean,7.772164e+01,2.275874e-01,8.990203e+02
std,2.498747e+02,1.277267e+00,3.829981e+03
min,1.000000e+00,0.000000e+00,0.000000e+00
25%,4.000000e+00,0.000000e+00,3.400000e+01
50%,1.600000e+01,0.000000e+00,1.600000e+02
75%,6.200000e+01,0.000000e+00,5.440000e+02
max,4.008400e+04,2.740000e+02,4.819460e+05


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal Test 1

**Hypothesis:** Pages with better search visibility have higher CTR.

**Test:** I divided webpages into three visibility groups using `NTILE(3)` based on `gsc_sum_position` and calculated the average CTR for each group.

**Result:** High Visibility = 0.004326, Medium Visibility = 0.002483, Low Visibility = 0.002433.

**Verdict:** **CONFIRMED.** The observed data shows that pages in higher visibility groups have higher average CTR than pages in lower visibility groups.

In [14]:
signal1 = con.sql(f"""
WITH position_groups AS (

SELECT
    gsc_clicks,
    gsc_impressions,
    gsc_sum_position,

    NTILE(3) OVER (
        ORDER BY gsc_sum_position
    ) AS position_group

FROM read_parquet(
'{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE
gsc_data_available IS TRUE
)

SELECT

CASE
    WHEN position_group = 1 THEN 'High Visibility'
    WHEN position_group = 2 THEN 'Medium Visibility'
    ELSE 'Low Visibility'
END AS visibility_group,

AVG(
CAST(gsc_clicks AS DOUBLE) /
NULLIF(gsc_impressions,0)
) AS avg_ctr,

COUNT(*) AS pages

FROM position_groups

GROUP BY position_group

ORDER BY position_group

""").df()

signal1


,visibility_group,avg_ctr,pages
0,High Visibility,0.004326,1203687
1,Medium Visibility,0.002483,1203687
2,Low Visibility,0.002433,1203687


### Signal Test 2

**Hypothesis:** Pages with better visibility receive more impressions.

**Test:** I grouped webpages into three visibility groups using `NTILE(3)` on `gsc_sum_position` and compared their average impressions.

**Result:** The Low Visibility group had the highest average impressions, while the High Visibility group had the lowest.

**Verdict:** **OPPOSITE.** This suggests that `gsc_sum_position` is not a direct measure of search ranking. Since it is a cumulative metric, pages with many impressions can also accumulate larger position sums. Therefore, this grouping should be interpreted carefully.

In [15]:
signal2 = con.sql(f"""
WITH position_groups AS (

SELECT
    gsc_impressions,
    gsc_sum_position,

    NTILE(3) OVER (
        ORDER BY gsc_sum_position
    ) AS position_group

FROM read_parquet(
'{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE gsc_data_available IS TRUE

)

SELECT

CASE
    WHEN position_group = 1 THEN 'High Visibility'
    WHEN position_group = 2 THEN 'Medium Visibility'
    ELSE 'Low Visibility'
END AS visibility_group,

AVG(gsc_impressions) AS avg_impressions,

COUNT(*) AS pages

FROM position_groups

GROUP BY position_group

ORDER BY position_group;

""").df()

signal2

,visibility_group,avg_impressions,pages
0,High Visibility,6.980261,1203687
1,Medium Visibility,27.590347,1203687
2,Low Visibility,198.594316,1203687


### Signal Test 3

**Hypothesis:** Pages with higher impressions receive more clicks.

**Test:** I divided webpages into three groups based on impressions using `NTILE(3)` and compared the average number of clicks.

**Result:** The average clicks increased from 0.0083 to 0.0452 to 0.6292 as average impressions increased.

**Verdict:** **CONFIRMED.** Pages with more impressions generally receive more clicks.

In [16]:
signal3 = con.sql(f"""
WITH impression_groups AS (

SELECT
    gsc_impressions,
    gsc_clicks,
    NTILE(3) OVER (ORDER BY gsc_impressions) AS impression_group

FROM read_parquet(
'{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE gsc_data_available IS TRUE

)

SELECT

impression_group,

AVG(gsc_impressions) AS avg_impressions,
AVG(gsc_clicks) AS avg_clicks,
COUNT(*) AS pages

FROM impression_groups

GROUP BY impression_group

ORDER BY impression_group

""").df()

signal3

,impression_group,avg_impressions,avg_clicks,pages
0,1,2.746633,0.008314,1203687
1,2,17.752726,0.045225,1203687
2,3,212.665566,0.629223,1203687


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## Flag-linked Test

**Chosen rule assumption:** Pages with better search visibility should achieve higher CTR.

**Test:** I grouped pages into three visibility groups using `gsc_sum_position` and compared their average CTR.

**Observation:** The High Visibility group had the highest average CTR, followed by Medium and Low Visibility.

**Conclusion:** The data supports this rule assumption. Better visibility was associated with higher CTR in this dataset, so this signal could be useful for identifying content that may benefit from optimization.

In [17]:
signal1 = con.sql(f"""
WITH position_groups AS (

SELECT
    gsc_clicks,
    gsc_impressions,
    gsc_sum_position,

    NTILE(3) OVER (
        ORDER BY gsc_sum_position
    ) AS position_group

FROM read_parquet(
'{DATASET}/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE
gsc_data_available IS TRUE
)

SELECT

CASE
    WHEN position_group = 1 THEN 'High Visibility'
    WHEN position_group = 2 THEN 'Medium Visibility'
    ELSE 'Low Visibility'
END AS visibility_group,

AVG(
CAST(gsc_clicks AS DOUBLE) /
NULLIF(gsc_impressions,0)
) AS avg_ctr,

COUNT(*) AS pages

FROM position_groups

GROUP BY position_group

ORDER BY position_group

""").df()

signal1


,visibility_group,avg_ctr,pages
0,High Visibility,0.004327,1203687
1,Medium Visibility,0.002482,1203687
2,Low Visibility,0.002434,1203687


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### ANSWER

The analysis suggests that webpages with better search visibility tend to achieve higher CTR, indicating that improving search visibility can increase user engagement. Content teams can prioritize optimizing pages with lower visibility to improve their ranking and potentially increase clicks. These findings provide decision-support for identifying pages that may benefit most from SEO improvements.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.